In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
import py3Dmol

## Train Machine Learning Models

- Calculate basic molecular properties of identified target-binding ligands using RDKit
- Additional data preparation
- Train machine learning models, e.g.:
    - Linear Regression
    - Ridge Regression
    - Random Forest (RF)
    - Support Vector Machine (SVM)
    - Gradient Boosting Machine (GBM)
    - Feed-forward Neural Network (FFNN)
    - Graph Neural Network (GNN)

In [ ]:
chembl_id = "CHEMBL402"

df = pd.read_csv(f"../data/{chembl_id}.csv").drop(columns=["Unnamed: 0"])
df

In [ ]:
# Add mol-type objects to dataframe for calculating molecular properties (see below)

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllColumns = True

# Add mol-type objects without H atoms
df["ROMol"] = df["canonical_smiles"].apply(Chem.MolFromSmiles)
# Add mol-type objects including H atoms
df["ROMolH"] = df["ROMol"].apply(Chem.AddHs)
# Generate 3D coordinates
df["ROMol3D"] = df.apply(lambda row: AllChem.EmbedMolecule(row["ROMolH"], randomSeed=42), axis=1)
df = df[df["ROMol3D"] == 0].reset_index().drop(columns=["index"])
# Optimize the 3D structure using the MMFF94 force field
df["ROMolOPT"] = df.apply(lambda row: AllChem.MMFFOptimizeMolecule(row["ROMolH"], maxIters=2000), axis=1)
df = df[df["ROMolOPT"] == 0].reset_index().drop(columns=["index"])
df

In [ ]:
# Calculate basic molecular properties of identified target-binding ligands using RDKit

from rdkit.Chem import Descriptors, Descriptors3D

df["MW"] = df["ROMol"].apply(Descriptors.MolWt)
df["logP"] = df["ROMol"].apply(Descriptors.MolLogP)
df["HBA"] = df["ROMol"].apply(Descriptors.NumHAcceptors)
df["HBD"] = df["ROMol"].apply(Descriptors.NumHDonors)

def Ro5(mw, logP, HBA, HBD) -> int:
    Ro5_counter = 0

    if mw <= 500:
      Ro5_counter += 1
    if logP <= 5:
      Ro5_counter += 1
    if HBA <= 10:
      Ro5_counter += 1
    if HBD <= 5:
      Ro5_counter += 1

    return Ro5_counter

df["Ro5"] = df.apply(lambda row: Ro5(row["MW"], row["logP"], row["HBA"], row["HBD"]), axis=1)
df["RB"] = df["ROMol"].apply(Descriptors.NumRotatableBonds)
df["TPSA"] = df["ROMol"].apply(Descriptors.TPSA)
df["ROG"] = df.apply(lambda row: Descriptors3D.RadiusOfGyration(row["ROMolH"]), axis=1)
df

In [ ]:
# Sanity check of all values!
df.describe()

In [ ]:
def display_molecule_3D(mol):
    """
    Display a 3D molecule using py3Dmol.
        Requires a molecule with 3D coordinates.
        mol: RDKit Mol object with 3D coordinates
    """
    mb = Chem.MolToMolBlock(mol)
    p = py3Dmol.view(width=400, height=400)
    p.addModel(mb, 'sdf')
    p.setStyle({'stick':{'colorscheme':'greyCarbon', 'radius': 0.1}, 'sphere':{'scale':0.25}})
    p.setBackgroundColor('0xeeeeee')
    p.zoomTo()
    display(p.show())

In [ ]:
display_molecule_3D(df["ROMolH"][0])